# Recs 007: User-facing qualitative evaluation

Purpose:
- Run a small, repeatable qualitative review of recommendation quality.
- Keep this separate from metric-heavy proxy notebooks (`recs_004`, `recs_006`).

Flow:
1. Load evaluation prompts.
2. Generate top-K recommendations with current default path (`raw_raw`).
3. Export a review sheet.
4. Fill manual judgments and summarize failure modes.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import pandas as pd

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root (no pyproject.toml). cwd={here}")

REPO_ROOT = _repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from steam_review_ml.recommender.retrieve import ContentRetriever

EVAL_JSONL = REPO_ROOT / "artifacts" / "recs" / "eval_queries_review_style.jsonl"
OUT_CSV = REPO_ROOT / "artifacts" / "recs" / "user_facing_qual_eval_sheet.csv"
OUT_SUMMARY_CSV = REPO_ROOT / "artifacts" / "recs" / "user_facing_qual_eval_summary.csv"

TOP_K = 10
print("Eval prompts:", EVAL_JSONL)
print("Output sheet:", OUT_CSV)

Eval prompts: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_queries_review_style.jsonl
Output sheet: /home/ryanr/workspace/steam_recommendations/artifacts/recs/user_facing_qual_eval_sheet.csv


In [2]:
def load_jsonl(path: Path) -> list[dict]:
    out: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

queries = load_jsonl(EVAL_JSONL)
retriever = ContentRetriever()
print("Loaded queries:", len(queries))
print("Indexed games:", len(retriever.index_frame))

Loaded queries: 31
Indexed games: 315


In [3]:
evaluation_rows: list[dict] = []
for query_row in queries:
    query_id = query_row.get("id", "")
    review_draft_text = query_row.get("review_draft", "")
    expected_theme_list = query_row.get("expected_themes", [])
    avoided_theme_list = query_row.get("avoid_themes", [])

    retrieval_hits = retriever.top_k(review_draft_text, k=TOP_K, structured=False)
    recommended_game_names = retrieval_hits["app_name"].tolist()

    evaluation_rows.append(
        {
            "id": query_id,
            "review_draft": review_draft_text,
            "expected_themes": " | ".join(expected_theme_list),
            "avoid_themes": " | ".join(avoided_theme_list),
            "top1": recommended_game_names[0] if recommended_game_names else "",
            "top3": " | ".join(recommended_game_names[:3]),
            "top10": " | ".join(recommended_game_names),
            "quality_label": "",  # manual: good / mixed / bad
            "failure_tags": "",   # manual: semicolon-separated tags
            "notes": "",          # manual free-text rationale
        }
    )

qualitative_eval_sheet = pd.DataFrame(evaluation_rows)
qualitative_eval_sheet.head(10)

2026-04-14 17:33:58.970630: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776202439.002996   79226 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776202439.013771   79226 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776202439.051951   79226 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776202439.052028   79226 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776202439.052031   79226 computation_placer.cc:177] computation placer alr

,id,review_draft,expected_themes,avoid_themes,top1,top3,top10,quality_label,failure_tags,notes
0,r001,"Combat feels amazing when parries click, and b...",challenging melee combat | boss-focused action...,easy casual gameplay,Titan Souls,Titan Souls | Sekiro™: Shadows Die Twice | Mom...,Titan Souls | Sekiro™: Shadows Die Twice | Mom...,,,
1,r002,Loved the farming loop and decorating my house...,cozy progression | farming/life sim | relaxing...,combat-heavy grind,Farm Together,Farm Together | Townscaper | House Flipper,Farm Together | Townscaper | House Flipper | A...,,,
2,r003,"The squad tactics are the best part: flanking,...",turn-based tactics | squad strategy | positioning,real-time twitch gameplay,Sniper Elite 4,Sniper Elite 4 | XCOM 2 | Tom Clancy's Rainbow...,Sniper Elite 4 | XCOM 2 | Tom Clancy's Rainbow...,,,
3,r004,Characters were great and I cared about their ...,narrative RPG | companion writing | choices/co...,checklist open world,Tales of Berseria,Tales of Berseria | Assassin's Creed Odyssey |...,Tales of Berseria | Assassin's Creed Odyssey |...,,,
4,r005,"Finished it in one weekend. Short, stylish, an...",short indie | atmospheric experience | concise...,very long runtime,FAR: Lone Sails,FAR: Lone Sails | Grimm's Hollow | Helltaker,FAR: Lone Sails | Grimm's Hollow | Helltaker |...,,,
5,r006,City planning and logistics are deep and satis...,city builder | management sim | optimization/l...,light arcade gameplay,Cities: Skylines,Cities: Skylines | Euro Truck Simulator 2 | Ba...,Cities: Skylines | Euro Truck Simulator 2 | Ba...,,,
6,r007,Gunplay is crisp and movement feels smooth. Ma...,competitive multiplayer FPS | tight gunplay | ...,single-player narrative focus,Due Process,Due Process | Tom Clancy's Rainbow Six Siege |...,Due Process | Tom Clancy's Rainbow Six Siege |...,,,
7,r008,"Piecing together clues was the highlight, and ...",investigation | mystery solving | deductive ga...,jump-scare horror,Outlast,Outlast | Phasmophobia | Little Nightmares,Outlast | Phasmophobia | Little Nightmares | D...,,,
8,r009,Runs are quick and builds can get wild. Some u...,roguelike structure | build variety | high rep...,slow linear progression,Nova Drift,Nova Drift | Dragon Cliff 龙崖 | Super Hexagon,Nova Drift | Dragon Cliff 龙崖 | Super Hexagon |...,,,
9,r010,Exploration is gorgeous and I kept getting dis...,open-world exploration | discovery | ambient a...,repetitive filler content,Townscaper,Townscaper | A Short Hike | FAR: Lone Sails,Townscaper | A Short Hike | FAR: Lone Sails | ...,,,


In [4]:
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
qualitative_eval_sheet.to_csv(OUT_CSV, index=False)
print("Wrote:", OUT_CSV)
print("\nHow to annotate:")
print("1) Open the CSV and fill quality_label with: good / mixed / bad")
print("2) Fill failure_tags with semicolon-separated tags, e.g. genre_mismatch;popularity_drift")
print("3) Add short notes explaining why")
print("4) Save CSV, then run the next cell to summarize")

Wrote: /home/ryanr/workspace/steam_recommendations/artifacts/recs/user_facing_qual_eval_sheet.csv
Fill quality_label, failure_tags, notes; then re-run the next cell.


In [5]:
# Re-load manually annotated sheet and summarize.
annotated_sheet = pd.read_csv(OUT_CSV)

quality_label_counts = annotated_sheet["quality_label"].fillna("").value_counts()
print("Quality label counts:")
display(quality_label_counts)

failure_tag_series = (
    annotated_sheet["failure_tags"].fillna("")
    .str.split(";")
    .explode()
    .str.strip()
)
failure_tag_series = failure_tag_series[failure_tag_series != ""]
failure_tag_counts = failure_tag_series.value_counts()
print("\nFailure tag counts:")
display(failure_tag_counts)

summary = pd.DataFrame({
    "metric": ["n_rows", "n_good", "n_mixed", "n_bad"],
    "value": [
        len(annotated_sheet),
        int((annotated_sheet["quality_label"] == "good").sum()),
        int((annotated_sheet["quality_label"] == "mixed").sum()),
        int((annotated_sheet["quality_label"] == "bad").sum()),
    ],
})
summary.to_csv(OUT_SUMMARY_CSV, index=False)
print("\nWrote:", OUT_SUMMARY_CSV)

Quality label counts:


quality_label
    31
Name: count, dtype: int64


Failure tag counts:


Series([], Name: count, dtype: int64)


Wrote: /home/ryanr/workspace/steam_recommendations/artifacts/recs/user_facing_qual_eval_summary.csv
